# Kelly vs. SBF gamblers

Same compounding gamble as `a_risky_gamble.ipynb`: each round, the underlying asset return is $R = 2$ or $R = 0.25$, each w.p. $\tfrac{1}{2}$. Now, instead of going all-in, each player bets a fraction $f$ of wealth each round, so wealth evolves as
$$W_{t+1} \;=\; W_t \bigl(1 + f\,(R - 1)\bigr).$$
We simulate $N = 10{,}000$ Kelly gamblers ($f^*_{\text{Kelly}} = 1/6$, the log-utility optimum) alongside $N = 10{,}000$ SBF gamblers ($f^*_{\text{SBF}} \approx 0.412$, the CRRA optimum at $\gamma = 0.75$, $K = 16$). The strategies are taken from `Kelly_etc.ipynb`; here we just play them forward and see where the players end up.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(544)

In [2]:
initial_investment = 1000
T = 100         # rounds per player
N = 10_000      # players per strategy
returns = np.array([2.0, 0.25])

f_kelly = 1 / 6
f_sbf   = 0.4121

In [3]:
# Vectorized Monte Carlo: each row is one player's T draws of the underlying asset.
draws = rng.choice(returns, size=(N, T), p=[0.5, 0.5])

kelly_multipliers = 1 + f_kelly * (draws - 1)
sbf_multipliers   = 1 + f_sbf   * (draws - 1)

kelly_final = initial_investment * np.prod(kelly_multipliers, axis=1)
sbf_final   = initial_investment * np.prod(sbf_multipliers,   axis=1)

## Expected final wealth: theory vs. simulation

Returns are i.i.d., so $E[W_T] = W_0 \cdot (E[\text{multiplier}])^T$. The arithmetic expected per-round multiplier under fraction $f$ is
$$E[1 + f(R-1)] \;=\; 1 + f\bigl(E[R] - 1\bigr) \;=\; 1 + 0.125\,f.$$

In [4]:
E_R = 0.5 * 2.0 + 0.5 * 0.25

def theoretical_E_WT(f):
    return initial_investment * (1 + f * (E_R - 1)) ** T

ev_table = pd.DataFrame({
    "Strategy":              ["Kelly (f = 1/6)",         "SBF (f = 0.4121)"],
    "Theoretical E[W_T]":    [f"${theoretical_E_WT(f_kelly):,.2f}",
                              f"${theoretical_E_WT(f_sbf):,.2f}"],
    "Simulated E[W_T]":      [f"${kelly_final.mean():,.2f}",
                              f"${sbf_final.mean():,.2f}"],
})
print(ev_table.to_string(index=False))

        Strategy Theoretical E[W_T] Simulated E[W_T]
 Kelly (f = 1/6)          $7,861.12        $7,551.32
SBF (f = 0.4121)        $151,860.20       $84,040.91


## Percentiles of the wealth distribution

In [5]:
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9, 99.99]

table = pd.DataFrame({
    "Percentile":     percentiles,
    "Kelly wealth":   [f"${v:,.2f}" for v in np.percentile(kelly_final, percentiles)],
    "SBF wealth":     [f"${v:,.2f}" for v in np.percentile(sbf_final,   percentiles)],
})
print(table.to_string(index=False))

 Percentile Kelly wealth     SBF wealth
       1.00      $118.42          $0.11
       5.00      $280.69          $0.96
      10.00      $499.01          $4.00
      25.00    $1,182.84         $34.16
      50.00    $2,803.77        $291.62
      75.00    $6,645.97      $2,489.55
      90.00   $15,753.41     $21,253.25
      95.00   $28,006.06     $88,775.76
      99.00   $66,384.74    $757,875.76
      99.90  $209,808.55 $13,223,188.81
      99.99  $373,005.41 $55,239,611.96


## Top 10 final wealth values among the 10,000 players (each strategy)

In [6]:
kelly_top10 = np.sort(kelly_final)[::-1][:10]
sbf_top10   = np.sort(sbf_final)[::-1][:10]

top10_table = pd.DataFrame({
    "Rank":         np.arange(1, 11),
    "Kelly wealth": [f"${v:,.2f}" for v in kelly_top10],
    "SBF wealth":   [f"${v:,.2f}" for v in sbf_top10],
})
print(top10_table.to_string(index=False))

 Rank Kelly wealth      SBF wealth
    1  $497,323.97 $112,885,935.53
    2  $372,992.98  $55,233,846.76
    3  $372,992.98  $55,233,846.76
    4  $279,744.73  $27,025,313.77
    5  $279,744.73  $27,025,313.77
    6  $279,744.73  $27,025,313.77
    7  $279,744.73  $27,025,313.77
    8  $279,744.73  $27,025,313.77
    9  $279,744.73  $27,025,313.77
   10  $209,808.55  $13,223,188.81


**Takeaway.** Both strategies share the same per-round positive arithmetic EV, but the distributions are nothing alike. Kelly's median player roughly triples starting wealth while the median SBF player loses about 70% of it. The SBF tail is far heavier — the top SBF player ends two orders of magnitude above the top Kelly player — so SBF's *arithmetic mean* exceeds Kelly's, even though Kelly dominates the SBF strategy at almost every percentile up to the 90th. Small differences in $f$ produce very different long-run distributions when wealth compounds over 100 rounds, and "expected wealth" hides the asymmetry.